# [Task Adherence Evaluator](https://learn.microsoft.com/en-us/python/api/azure-ai-evaluation/azure.ai.evaluation.taskadherenceevaluator?view=azure-python)

**IMPORTANT NOTE**<br/>
- These samples use `GPT-4.1 mini` because `azure-ai-evaluation 1.18.3` local agent evaluators send the legacy max_tokens parameter.
- Newer GPT-5 deployments require `max_completion_tokens` and aren't compatible with this local evaluator path.
- For managed evaluations with newer judge models, see 4 - cloud evaluation.

## Objective
This sample demonstrates to how to use task adherence evaluator on agent data. The supported input formats include:
- simple data such as strings;
- user-agent conversations in the form of list of agent messages.

## What this evaluator assesses
The Task Adherence evaluator measures how well the agent adheres to their assigned tasks or predefined goal.

The scoring is on a 1-5 integer scale and is as follows:
  - Score 1: Fully Inadherent
  - Score 2: Barely Adherent
  - Score 3: Moderately Adherent
  - Score 4: Mostly Adherent
  - Score 5: Fully Adherent

The evaluation requires the following inputs:
  - Query    : The user query. Either a string with a user request or a list of messages with previous requests from the user and responses from the assistant, potentially including a system message.
  - Response : The response to be evaluated. Either a string or a message with the response from the agent to the last user query.

There is a third optional parameter:
  - ToolDefinitions : The list of tool definitions the agent can call. This may be useful for the evaluator to better assess if the right tool was called to adhere to user intent.

## Variables, Constants and Libraries definition

In [1]:
import os, sys
from dotenv import load_dotenv  # requires python-dotenv
from azure.identity import DefaultAzureCredential

if not load_dotenv():
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

openai_api_version = os.environ["AZURE_OPENAI_API_VERSION"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
foundry_project_endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT")

# gpt-4.1-mini is the latest working model because later models require max_completion_tokens, while this evaluator sends max_tokens
azure_evaluation_compatible_deployment_name = os.environ[
    "AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME"
]

credential = DefaultAzureCredential(
    exclude_environment_credential=True
)

print(f"azure_openai_endpoint: {azure_openai_endpoint}")
print(f"foundry_project_endpoint: {foundry_project_endpoint}")
print(f"azure_evaluation_compatible_deployment_name: {azure_evaluation_compatible_deployment_name}")
print(f"openai_api_version: {openai_api_version}")

azure_openai_endpoint: https://mm-ai-upskilling-project-resourc.openai.azure.com/
foundry_project_endpoint: https://mm-ai-upskilling-project-resourc.services.ai.azure.com/api/projects/ai-upskilling-project
azure_evaluation_compatible_deployment_name: gpt-4.1-mini
openai_api_version: 2025-04-01-preview


### Initialize Task Adherence Evaluator


In [2]:
from azure.ai.evaluation import TaskAdherenceEvaluator, AzureOpenAIModelConfiguration
from pprint import pprint
import warnings

warnings.filterwarnings("ignore")

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment=azure_evaluation_compatible_deployment_name,
    api_version=openai_api_version,
)

task_adherence_evaluator = TaskAdherenceEvaluator(model_config, credential=credential)

# Print some constants
print(f'openai endpoint: <{model_config["azure_endpoint"]}>')
print(f'azure deployment name: <{model_config["azure_deployment"]}>')
print(f'openai api version: <{model_config["api_version"]}>')

Class TaskAdherenceEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


openai endpoint: <https://mm-ai-upskilling-project-resourc.openai.azure.com/>
azure deployment name: <gpt-4.1-mini>
openai api version: <2025-04-01-preview>


### Samples

#### Evaluating query and response as string

In [3]:
# Failure example, there's only a vague adherence to the task
result = task_adherence_evaluator(
    query="What are the best practices for maintaining a healthy rose garden during the summer?",
    response="Make sure to water your roses regularly and trim them occasionally.",
)
pprint(result)

Conversation history could not be parsed; falling back to raw input. Evaluator accuracy will degrade. Input shape: type=str len=84. Error: 'str' object has no attribute 'get'
Agent response could not be parsed, falling back to original response. Error: 'str' object has no attribute 'get'. type=str len=67


{'task_adherence': 0.0,
 'task_adherence_passed': False,
 'task_adherence_properties': {'completion_tokens': 205,
                               'finish_reason': 'stop',
                               'model': 'gpt-4.1-mini-2025-04-14',
                               'prompt_tokens': 1473,
                               'sample_input': '[{"role": "user", "content": '
                                               '"{\\"system_message\\": '
                                               '\\"\\", \\"query\\": \\"What '
                                               'are the best practices for '
                                               'maintaining a healthy rose '
                                               'garden during the summer?\\", '
                                               '\\"response\\": \\"Make sure '
                                               'to water your roses regularly '
                                               'and trim them '
                    

In [4]:
# Success example, full adherence to the task
result = task_adherence_evaluator(
    query="What are the best practices for maintaining a healthy rose garden during the summer?",
    response="For optimal summer care of your rose garden, start by watering deeply early in the morning to ensure the roots are well-hydrated without encouraging fungal growth. Apply a 2-3 inch layer of organic mulch around the base of the plants to conserve moisture and regulate soil temperature. Fertilize with a balanced rose fertilizer every 4 to 6 weeks to support healthy growth. Prune away any dead or diseased wood to promote good air circulation, and inspect regularly for pests such as aphids or spider mites, treating them promptly with an appropriate organic insecticidal soap. Finally, ensure that your roses receive at least 6 hours of direct sunlight daily for robust flowering.",
)
pprint(result)

Conversation history could not be parsed; falling back to raw input. Evaluator accuracy will degrade. Input shape: type=str len=84. Error: 'str' object has no attribute 'get'
Agent response could not be parsed, falling back to original response. Error: 'str' object has no attribute 'get'. type=str len=681


{'task_adherence': 1.0,
 'task_adherence_passed': True,
 'task_adherence_properties': {'completion_tokens': 135,
                               'finish_reason': 'stop',
                               'model': 'gpt-4.1-mini-2025-04-14',
                               'prompt_tokens': 1597,
                               'sample_input': '[{"role": "user", "content": '
                                               '"{\\"system_message\\": '
                                               '\\"\\", \\"query\\": \\"What '
                                               'are the best practices for '
                                               'maintaining a healthy rose '
                                               'garden during the summer?\\", '
                                               '\\"response\\": \\"For optimal '
                                               'summer care of your rose '
                                               'garden, start by watering '
            

#### Evaluating query and response as list of messages

In [5]:
query = [
    {"role": "system", "content": "You are an expert in literature and at provid can provide book recommendations."},
    {
        "createdAt": "2025-03-14T08:00:00Z",
        "role": "user",
        "content": [
            {"type": "text", "text": "I love historical fiction. Can you recommend a good book from that genre?"}
        ],
    },
]

response = [
    {
        "createdAt": "2025-03-14T08:00:05Z",
        "role": "assistant",
        "content": [{"type": "text", "text": "Let me fetch a recommendation for historical fiction."}],
    },
    {
        "createdAt": "2025-03-14T08:00:10Z",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250314_001",
                "name": "get_book",
                "arguments": {"genre": "historical fiction"},
            }
        ],
    },
    {
        "createdAt": "2025-03-14T08:00:15Z",
        "role": "tool",
        "tool_call_id": "tool_call_20250314_001",
        "content": [
            {
                "type": "tool_result",
                "tool_result": '{ "book": { "title": "The Pillars of the Earth", "author": "Ken Follett", "summary": "A captivating tale set in medieval England that weaves historical events with personal drama." } }',
            }
        ],
    },
    {
        "createdAt": "2025-03-14T08:00:20Z",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Based on our records, I recommend 'The Pillars of the Earth' by Ken Follett. This novel is an excellent example of historical fiction with a rich narrative and well-developed characters. Would you like more details or another suggestion?",
            }
        ],
    },
]

tool_definitions = [
    {
        "name": "get_book",
        "description": "Retrieve a book recommendation for a specified genre.",
        "parameters": {
            "type": "object",
            "properties": {
                "genre": {"type": "string", "description": "The genre for which a book recommendation is requested."}
            },
        },
    }
]

result = task_adherence_evaluator(
    query=query,
    response=response,
    tool_definitions=tool_definitions,
)
pprint(result)

{'task_adherence': 1.0,
 'task_adherence_passed': True,
 'task_adherence_properties': {'completion_tokens': 135,
                               'finish_reason': 'stop',
                               'model': 'gpt-4.1-mini-2025-04-14',
                               'prompt_tokens': 1607,
                               'sample_input': '[{"role": "user", "content": '
                                               '"{\\"system_message\\": '
                                               '\\"\\", \\"query\\": '
                                               '\\"SYSTEM_PROMPT:\\\\n  You '
                                               'are an expert in literature '
                                               'and at provid can provide book '
                                               'recommendations.\\\\n\\\\nUser '
                                               'turn 1:\\\\n  I love '
                                               'historical fiction. Can you '
                   